# Imports

In [ ]:
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt

# Data Loading

## Cohort

In [ ]:
data_path = "data/cohort/cohort_icu_mortality_0_.csv.gz"
cohort = pd.read_csv(data_path, compression="gzip")

In [ ]:
cohort.info()

In [ ]:
cohort.head()

## Features

In [ ]:
features_dir = "data/features"

feature_files = glob.glob(os.path.join(features_dir, "*.csv")) + \
                glob.glob(os.path.join(features_dir, "*.csv.gz"))

features = {}
for path in feature_files:
    print(f"Processing {path}")
    name = os.path.basename(path)
    if name.endswith(".csv.gz"):
        key = name[:-7]
    elif name.endswith(".csv"):
        key = name[:-4]
    else:
        print(f"Unknown file extension: {name}")
        key = name
    features[key] = pd.read_csv(path, compression="gzip" if path.endswith(".gz") else None)

In [ ]:
for df_key in features:
    print(f"{df_key}:")
    df = features[df_key]
    display(df.head())

# Map Names

In [ ]:
mimic = "mimiciv/3.1"
d_items = pd.read_csv(f"{mimic}/icu/d_items.csv.gz", compression="gzip", usecols=["itemid", "label"])
d_icd = pd.read_csv(f"{mimic}/hosp/d_icd_diagnoses.csv.gz", compression="gzip", usecols=["icd_code", "long_title"])
item_to_label = d_items.set_index("itemid")["label"]
icd_to_title = d_icd.drop_duplicates(subset="icd_code").set_index("icd_code")["long_title"]

In [ ]:
features["preproc_med_icu"]["item_label"] = features["preproc_med_icu"]["itemid"].map(item_to_label)
features["preproc_chart_icu"]["item_label"] = features["preproc_chart_icu"]["itemid"].map(item_to_label)
features["preproc_out_icu"]["item_label"] = features["preproc_out_icu"]["itemid"].map(item_to_label)
features["preproc_proc_icu"]["item_label"] = features["preproc_proc_icu"]["itemid"].map(item_to_label)
features["preproc_diag_icu"]["diag_name"] = features["preproc_diag_icu"]["new_icd_code"].map(icd_to_title)

In [ ]:
for df_key in features:
    print(f"{df_key}:")
    df = features[df_key]
    display(df.head())

# EDA

## Cohort

In [ ]:
n_unique_subjects = cohort["subject_id"].nunique()
print(f"Number of unique subjects: {n_unique_subjects}")

In [ ]:
n_unique_hadm_ids = cohort["hadm_id"].nunique()
print(f"Number of unique hadm_id: {n_unique_hadm_ids}")

In [ ]:
n_unique_stays = cohort["stay_id"].nunique()
print(f"Number of unique stay_id: {n_unique_stays}")

In [ ]:
n_deaths_dod = (cohort["dod"] != "0").sum()
percent_deaths_dod = n_deaths_dod / len(cohort) * 100
print(f"Number of cohort stays with DOD (date of death) not equal to '0': {n_deaths_dod} ({percent_deaths_dod:.2f}%)")

In [ ]:
n_deaths = cohort["label"].sum()
percent_deaths = n_deaths / len(cohort) * 100
print(f"Number of deaths: {n_deaths} ({percent_deaths:.2f}%)")

In [ ]:
sex_counts = cohort["gender"].value_counts()
sex_counts.plot(kind="bar", figsize=(6,4), title="Sex Distribution")
plt.xlabel("Sex")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
cohort["Age"].hist(bins=30)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Unique Items

In [ ]:
for df_key in features:
    print(f"{df_key}:")
    df = features[df_key]
    if "itemid" in df.columns:
        print(f"Unique itemids: {df['itemid'].nunique()}")
    if "new_icd_code" in df.columns:
        print(f"Unique new_icd_codes: {df['new_icd_code'].nunique()}")
    print("")

## Most Frequent

In [ ]:
N = 20 # top N to show

def top_bar(series, title, xlabel="Count"):
    ax = series.head(N).plot.barh(figsize=(8, min(10, N * 0.4)), title=title, xlabel="")
    ax.set_xlabel(xlabel)
    plt.tight_layout()
    plt.show()

In [ ]:
top_bar(
    features["preproc_med_icu"]["item_label"].value_counts(),
    "Top medications (preproc_med_icu)"
)

In [ ]:
top_bar(
    features["preproc_chart_icu"]["item_label"].value_counts(),
    "Top chart items (preproc_chart_icu)"
)

In [ ]:
top_bar(
    features["preproc_out_icu"]["item_label"].value_counts(),
    "Top output items (preproc_out_icu)"
)

In [ ]:
top_bar(
    features["preproc_proc_icu"]["item_label"].value_counts(),
    "Top procedures (preproc_proc_icu)"
)

In [ ]:
top_bar(
    features["preproc_diag_icu"]["diag_name"].value_counts(),
    "Top diagnoses (preproc_diag_icu)"
)